In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

from itertools import product

import pandas as pd
from strat import STRAT

from dema_strat import (
  compare_benchmark,
  db_connect,
  global_style,
  grid_tuning,
  load_data_and_split,
)
from exits_builder import build_be_exits
from sizer import sl_based_sizer
from visualizer import (
  combinations_compare_visualizer,
  portfolio_perf_visualizer,
  trades_visualizer,
)

SPLIT_RATIO = 0.8
TABLE_NAME = 'f0'
TABLE_NAME_OOS = f"{TABLE_NAME}_OOS"
DATA_PATH = '~/market_data/XAUUSD/XAUUSD_M15.csv'

PARAMS = {
    "ema_period": {"type": "INTEGER", "data": range(5, 15, 2)},
    "smoothing_period": {"type": "INTEGER", "data": range(6, 16, 2)},
    "use_vwap": {"type": "BOOL", "data": [True, False]},
    "atr_period": {"type": "INTEGER", "data": range(14, 22, 2)},
    "sl_mult": {"type": "REAL", "data": range(1,3)},
    "rr_ratio": {"type": "REAL", "data": [0.25, 0.5, 1.0, 1.5, 2.0]},
    "asset": {"type": "TEXT", "data": ["XAUUSD"]},
    "timeframe": {"type": "TEXT", "data": ["M15"]},
}

PARAMS_GRID = list(product(*(v["data"] for v in PARAMS.values())))
TOP_N = max(100, len(PARAMS_GRID))

conn, cursor = db_connect()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Benchmark

In [6]:
df_ios, df_oos = load_data_and_split(DATA_PATH, split_ratio=SPLIT_RATIO)

def run_backtest(data, params = {}):  # noqa: B006
  # build the strat
  strat = STRAT(data, params)

  rr_ratio = params.get("rr_ratio", 2)
  sl_mult = params.get("sl_mult", 1)

  # exits and sizer
  long_exits, short_exits, sl_dist = build_be_exits(data, strat.signals, rr_ratio=rr_ratio, sl_mult=sl_mult)
  sizer = sl_based_sizer(risk_per_trade=10, sl_dists=sl_dist)

  portfolio = strat._run_backtest(long_exits, short_exits, size=sizer, size_type="amount")
  return portfolio, strat

portfolio, strat = run_backtest(df_ios, {
  "ema_period": 9,
  "smoothing_period": 12,
  "use_vwap": True,
  "rr_ratio": 1.5,
  "sl_mult": 2,
  "atr_period": 16,
})
compare_benchmark(portfolio, df_ios, freq=strat.freq)

[loader] cache hit: XAUUSD_M15__cache_v1.parquet


/home/chris/backtester/dema_strat/src/dema_strat/utils.py:63: UserWarning: Object has multiple columns. Aggregating using <function mean at 0x7cb505be3060>. Pass column to select a single column/group.
  {"STRATEGY": portfolio.stats(), "BENCHMARK": pf_benchmark.stats()}


,STRATEGY,BENCHMARK
Total Return [%],-10.404,50.593
Max Drawdown [%],10.705,34.888
Sharpe Ratio,-4.673,1.188
Sortino Ratio,-6.510,N/A
Calmar Ratio,-0.316,0.914
Profit Factor,0.709,N/A
Total Trades,3700,1
Win Rate [%],37.459,N/A
Total Fees Paid,"5,312.497",0


## Trade and Portfolio performance

In [ ]:
trades_visualizer(data=df_ios, signals=strat.signals, portfolio=portfolio, N = 500)
portfolio_perf_visualizer(portfolio)

## Grid tuning

In [ ]:
grid_tuning(params=PARAMS, table_name=TABLE_NAME, data=df_ios, run_backtest=run_backtest)
grid_tuning(params=PARAMS, table_name=f"{TABLE_NAME}_OOS", data=df_oos, run_backtest=run_backtest)

## Combinations compare for TOP 200 vs All combinations

In [ ]:
SORT_BY = "profit_factor"

def get_query(limit = True):
    return f"""
        SELECT *
        FROM {TABLE_NAME}
        ORDER BY {SORT_BY} DESC
        {f'LIMIT {TOP_N}' if limit else ''}
    """

top_df = pd.read_sql_query(get_query(), conn)
all_df = pd.read_sql_query(get_query(limit = False), conn)

print(f"Top {TOP_N} combinations")
display(top_df)

combinations_compare_visualizer(
  first_data=top_df,
  second_data=all_df,
  chart_title=f"TOP {TOP_N} vs All combinations",
  first_name="TOP 200",
  second_name="All combinations"
)

Top 200 combinations


,id,name,ema_period,smoothing_period,use_vwap,atr_period,sl_mult,rr_ratio,asset,timeframe,profit_factor,winrate,expectancy,max_dd,sharp_ratio,sortino_ratio,calmar_ratio,return_pct,n_trades,fee
0,1560,11_14_True_20_2_2.0_XAUUSD_M15,11,14,1,20,2.0,2.0,XAUUSD,M15,0.757862,32.684584,-1.315350,8.915322,-3.518398,-4.968663,-0.308195,-8.515574,3237.0,4572.666744
1,1880,13_12_True_20_2_2.0_XAUUSD_M15,13,12,1,20,2.0,2.0,XAUUSD,M15,0.754845,32.652430,-1.334844,9.011848,-3.555334,-5.021278,-0.308965,-8.625764,3231.0,4560.777146
2,1550,11_14_True_18_2_2.0_XAUUSD_M15,11,14,1,18,2.0,2.0,XAUUSD,M15,0.748772,32.653692,-1.367265,9.264189,-3.655558,-5.159058,-0.308678,-8.851675,3237.0,4591.029086
3,1870,13_12_True_18_2_2.0_XAUUSD_M15,13,12,1,18,2.0,2.0,XAUUSD,M15,0.747631,32.704208,-1.374521,9.275635,-3.662395,-5.172642,-0.309492,-8.884906,3232.0,4579.958426
4,1540,11_14_True_16_2_2.0_XAUUSD_M15,11,14,1,16,2.0,2.0,XAUUSD,M15,0.744458,32.506972,-1.394912,9.409812,-3.717837,-5.229471,-0.309261,-9.002764,3227.0,4595.638348
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,829,9_6_True_18_2_1.5_XAUUSD_M15,9,6,1,18,2.0,1.5,XAUUSD,M15,0.696322,35.596330,-1.417816,12.718025,-5.328162,-7.460454,-0.318215,-12.363356,4360.0,6292.091103
196,1519,11_12_False_20_2_1.5_XAUUSD_M15,11,12,0,20,2.0,1.5,XAUUSD,M15,0.695852,36.539379,-1.417121,11.952512,-5.134156,-6.960874,-0.324632,-11.875473,4190.0,5834.410332
197,280,5_12_True_20_2_2.0_XAUUSD_M15,5,12,1,20,2.0,2.0,XAUUSD,M15,0.695782,29.475542,-1.593858,13.088353,-4.955097,-6.845664,-0.316529,-12.642485,3966.0,5677.910948
198,1809,13_10_False_14_2_1.5_XAUUSD_M15,13,10,0,14,2.0,1.5,XAUUSD,M15,0.695777,36.649215,-1.430921,12.095316,-5.149565,-6.972499,-0.325035,-12.025458,4202.0,5973.003820


## Combinations compare for OOS vs IOS

In [ ]:
SORT_BY = "profit_factor"

def get_query(table):
    return f"""
        SELECT *
        FROM {table}
        ORDER BY {SORT_BY} DESC
        LIMIT {TOP_N}
    """

ios = pd.read_sql_query(get_query(TABLE_NAME), conn)
oos = pd.read_sql_query(get_query(f"{TABLE_NAME}_OOS"), conn)

print(f"OOS combinations (TOP {TOP_N})")
display(oos)

combinations_compare_visualizer(
  first_data=oos,
  second_data=ios,
  chart_title=f"OOS vs IOS (TOP {TOP_N})",
  first_name="OOS",
  second_name="IOS"
)

## Cluster parameter combination

In [17]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_sql_query(f"""
    SELECT *
    FROM {TABLE_NAME}
""", conn)

features = ['profit_factor', 'winrate', 'calmar_ratio', 'sortino_ratio'] 
TARGET = 'profit_factor'   # the metric you want to optimise for
param_cols = [p for p in PARAMS if p not in ('asset', 'timeframe')]

# clean: inf from zero-loss / zero-downside combos
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=features)

def find_param_plateaus(g, top_q=0.90, k_range=range(2, 10), target=TARGET):
    top = g[g[target] >= g[target].quantile(top_q)]
    if len(top) < 20:
        return None
    X = StandardScaler().fit_transform(top[param_cols])

    best = max(
        (
            (silhouette_score(X, lab), k, lab)
            for k, lab in (
                (k, KMeans(k, init='k-means++', n_init=10, random_state=42).fit_predict(X))
                for k in k_range if k < len(top)
            )
        ),
        key=lambda t: t[0],
    )
    score, k, labels = best
    top = top.assign(cluster_id=labels)

    prof = top.groupby('cluster_id').agg(
        size=(target, 'size'),
        median_target=(target, 'median'),
        q25_target=(target, lambda s: s.quantile(0.25)),
        **{c: (c, 'mean') for c in features},
        **{c: (c, 'median') for c in param_cols},
    )
    # rank by worst-case, not best-case: a plateau you can actually trade
    return prof.sort_values(['q25_target', 'size'], ascending=False), score, k

# for (asset, tf), g in df.groupby(['asset', 'timeframe']):
#     res = find_param_plateaus(g)
#     if res is None:
#         continue
#     prof, score, k = res
#     print(f"{asset} {tf} — k={k}, silhouette={score:.3f}, TARGET: {TARGET}")
#     print(f"*All metrics are mean values {features}")
#     display(global_style(prof))

for target in features:
    res = find_param_plateaus(g=df, target=target)
    if res is None:
        continue
    prof, score, k = res
    print(f"k={k}, silhouette={score:.3f}, TARGET: {target}")
    print(f"*All metrics are mean values {features}")
    display(global_style(prof))

k=8, silhouette=0.257, TARGET: profit_factor
*All metrics are mean values ['profit_factor', 'winrate', 'calmar_ratio', 'sortino_ratio']


size,median_target,q25_target,profit_factor,winrate,calmar_ratio,sortino_ratio,ema_period,smoothing_period,use_vwap,atr_period,sl_mult,rr_ratio
29,0.73,0.73,0.73,32.28,-0.31,-5.49,11.00,12.00,1.00,18.00,2.00,2.00
31,0.71,0.70,0.71,37.35,-0.32,-6.53,9.00,12.00,1.00,16.00,2.00,1.50
27,0.71,0.70,0.71,37.22,-0.32,-6.58,13.00,8.00,1.00,16.00,2.00,1.50
20,0.71,0.70,0.71,31.08,-0.32,-6.24,11.00,13.00,0.00,16.00,2.00,2.00
18,0.71,0.70,0.71,30.12,-0.32,-6.80,11.00,10.00,0.00,20.00,2.00,2.00
27,0.71,0.70,0.71,30.81,-0.31,-6.33,11.00,8.00,1.00,16.00,2.00,2.00
29,0.70,0.70,0.71,30.36,-0.31,-6.50,7.00,10.00,1.00,18.00,2.00,2.00
19,0.70,0.70,0.70,36.42,-0.32,-6.91,11.00,12.00,0.00,18.00,2.00,1.50


k=3, silhouette=0.314, TARGET: winrate
*All metrics are mean values ['profit_factor', 'winrate', 'calmar_ratio', 'sortino_ratio']


size,median_target,q25_target,profit_factor,winrate,calmar_ratio,sortino_ratio,ema_period,smoothing_period,use_vwap,atr_period,sl_mult,rr_ratio
96,47.13,46.11,0.50,47.16,-0.33,-12.42,9.00,10.00,1.00,17.00,2.00,0.50
64,46.37,45.63,0.49,46.51,-0.33,-13.79,11.00,11.00,0.00,17.00,2.00,0.50
40,45.85,45.53,0.65,45.90,-0.32,-8.09,11.00,12.00,1.00,17.00,2.00,1.00


k=2, silhouette=0.445, TARGET: calmar_ratio
*All metrics are mean values ['profit_factor', 'winrate', 'calmar_ratio', 'sortino_ratio']


size,median_target,q25_target,profit_factor,winrate,calmar_ratio,sortino_ratio,ema_period,smoothing_period,use_vwap,atr_period,sl_mult,rr_ratio
185,-0.32,-0.32,0.71,33.81,-0.31,-6.52,9.00,10.00,1.00,18.00,2.00,2.00
15,-0.32,-0.32,0.72,32.66,-0.32,-6.00,13.00,14.00,0.00,16.00,2.00,2.00


k=7, silhouette=0.282, TARGET: sortino_ratio
*All metrics are mean values ['profit_factor', 'winrate', 'calmar_ratio', 'sortino_ratio']


size,median_target,q25_target,profit_factor,winrate,calmar_ratio,sortino_ratio,ema_period,smoothing_period,use_vwap,atr_period,sl_mult,rr_ratio
30,-5.45,-5.61,0.73,32.24,-0.31,-5.50,11.00,12.00,1.00,16.00,2.00,2.00
27,-6.33,-6.61,0.71,37.56,-0.32,-6.40,11.00,12.00,1.00,20.00,2.00,1.50
35,-6.52,-6.71,0.71,30.64,-0.31,-6.41,11.00,8.00,1.00,18.00,2.00,2.00
27,-6.51,-6.79,0.71,37.42,-0.31,-6.54,11.00,12.00,1.00,16.00,2.00,1.50
33,-6.41,-6.79,0.71,30.90,-0.32,-6.37,11.00,12.00,0.00,18.00,2.00,2.00
29,-6.64,-6.79,0.70,30.22,-0.31,-6.59,5.00,12.00,1.00,16.00,2.00,2.00
19,-6.75,-6.95,0.70,36.78,-0.32,-6.77,13.00,12.00,0.00,16.00,2.00,1.50
